# From solar images to filament masks
**Author:** Shep Scientific. **Team:** shepsci.

This notebook teaches and reproduces our final two-model method. A mask is the set of image pixels assigned to one filament. The models propose masks. We combine those proposals, remove duplicates, assign shared pixels, and encode the final instances.

Use the exact weights stored in this public repository. Git LFS restores both model files. Start with README.md for setup and SECURITY.md for environment guidance. NOTICE credits HDJoJo for the original released weights and initial inference design. The report and provenance.json document the model source and recorded training settings.

## 1. Prepare the software environment
The package versions define the numerical implementation. The recorded replay uses Python 3.12.12 and Apple MPS. requirements.txt lists all package versions. These versions preserve the submitted environment; SECURITY.md documents known dependency advisories. Use a dedicated environment. README describes storage and compute requirements.

In [ ]:
%pip install -r requirements.txt

## 2. Locate the official images and our stored models
Set IMAGES to the official test/test_images directory. The file loader checks each model file against its SHA256 hash. A matching hash means that you have the exact recorded bytes. architecture.json describes the networks stored in those files.

The image predictor reads three identical grayscale channels and divides integer pixel values by 255. For example, intensity 128 becomes 0.502. At a target dimension of 2048, a square 2048-pixel input keeps its geometry. Convolutional layers extract features at several scales. A segmentation head combines shared mask patterns with per-instance coefficients.

In [ ]:
from pathlib import Path
import json
import inference
IMAGES = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images")
TRAIN = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train")
OUTPUT = Path("work")
assert IMAGES.is_dir(), "Set IMAGES to official test/test_images"
weights = inference.restore_weights()
print("Owned local weights:", weights)
print(json.loads(Path("architecture.json").read_text())[0]["head_class"])

## Worked mask example
A has 40 pixels and B has 30. They share 10 pixels, so IoU is 10 / (40 + 30 - 10) = 1/6. Both pass duplicate removal. A claims pixels first and B keeps 20 remaining pixels. Candidate D overlaps A by 35 of 45 union pixels, so its IoU exceeds 0.5 and it is removed. This cell uses the same inference functions as the final method.

In [ ]:
import numpy as np
# Synthetic pixels make the mask operations visible without loading image data.
tutorial_a = np.zeros((2048, 2048), dtype=np.uint8)
tutorial_b = np.zeros_like(tutorial_a)
tutorial_duplicate = np.zeros_like(tutorial_a)
tutorial_a[2:7, 2:10] = 1       # 40 pixels, confidence 0.90
tutorial_b[5:10, 5:11] = 1      # 30 pixels, confidence 0.80
tutorial_duplicate[2:7, 3:11] = 1  # 40 pixels, confidence 0.70
tutorial_shared = int((tutorial_a & tutorial_b).sum())
tutorial_iou = tutorial_shared / int((tutorial_a | tutorial_b).sum())
print({"shared_pixels": tutorial_shared, "IoU_A_B": tutorial_iou})
tutorial_candidates = [
    {"rle": inference.rle(tutorial_a), "score": 0.90},
    {"rle": inference.rle(tutorial_b), "score": 0.80},
    {"rle": inference.rle(tutorial_duplicate), "score": 0.70},
]
tutorial_final = [inference.decode(x) for x in inference.paint_and_nms(tutorial_candidates)]
assert len(tutorial_final) == 2
assert [int(x.sum()) for x in tutorial_final] == [40, 20]
assert not (tutorial_final[0] & tutorial_final[1]).any()
print("Final instance areas:", [int(x.sum()) for x in tutorial_final])


## 3. Apply the final method to all supplied images
Each model predicts masks and confidence scores. Both use image dimension 2048, confidence threshold 0.5, box-IoU argument 0, and maximum 100 detections. The YOLO26 end-to-end head retains its pinned library behavior.

We pool candidates in confidence order. Mask IoU above 0.5 identifies a duplicate of a higher-priority mask. The remaining masks claim pixels in the same order. We keep regions with at least five pixels after subtraction. inference.py implements these reusable steps. The next cell runs the entire test set and writes compressed COCO RLE.

In [ ]:
receipt = inference.main(IMAGES, OUTPUT / "submission.csv")
receipt

## 4. Read the output checks
A valid submission assigns each pixel to at most one filament. The validator decodes every mask, checks dimensions, and checks identifiers and pixel separation. An image with no detected filament emits no rows.

The final result contains 1,317 masks in 178 of 180 processed images. The displayed public PQ is 0.40. The recorded SHA256 identifies the exact selected CSV. The cell below compares your output with that receipt. Different processing backends can change predictions near numerical thresholds.

In [ ]:
from validate import check
assert check(OUTPUT / "submission.csv") == 0
reference = json.loads(Path("validation.json").read_text())
same = receipt["sha256"] == reference["receipt"]["sha256"]
print({"matches_recorded_csv": same, "device": receipt["device"]})
if not same:
    print("Investigate backend and version differences before claiming exact replay.")

## 5. Learn from diagnostic measures and component baselines
PQ combines instance detection and shape agreement. A match has IoU above 0.5. Extra predictions count as false positives; missed reference instances count as false negatives. The denominator is TP + 0.5 FP + 0.5 FN.

Binary Dice and IoU compare the union of predicted pixels with the annotator's union mask. One-to-many and many-to-one counts describe split and merged instances at the documented diagnostic overlap threshold 0.1.

The report compares the final method with each individual model on the same 12 official training observations. The fixed hash rule selects those observations before inference scores are computed. Their 17 annotation records stay grouped by image during bootstrap sampling. The included weights have seen these training observations, so these results are descriptive training diagnostics. diagnostics.json records the results and uncertainty intervals. Set RUN_DIAGNOSTICS to True to reproduce the final-method diagnostic values.

In [ ]:
RUN_DIAGNOSTICS = False
if RUN_DIAGNOSTICS:
    import evaluate
    diagnostic = json.loads(Path("diagnostics.json").read_text())
    selected = diagnostic["observations"]
    paths = [TRAIN / "train_images" / f"{s}.jpeg" for s in selected]
    assert all(p.is_file() for p in paths)
    from ultralytics import YOLO
    models = [YOLO(str(p)) for p in weights]
    predictions = {s: inference.paint_and_nms(sum([inference.predict(m, p, device=receipt["device"]) for m in models], [])) for s, p in zip(selected, paths)}
    destination = OUTPUT / "training-diagnostic.csv"
    inference.write_submission(destination, predictions)
    metrics = evaluate.evaluate(destination, TRAIN / "MAGFiLO_1.0_Annotations_kaggle2026_train.json", selected)
    print({k:v for k,v in metrics.items() if k != "per_record"})

## 6. Optional learning path: train a new model
Model weights are numerical values adjusted during training. train.py exports each official instance polygon as a filament example and creates a fixed observation-grouped split. All annotation records from one observation stay together. This gives 566 training and 141 validation observations.

The optional procedure starts a fresh network with our recorded architecture. It uses seed 3407, AdamW, learning rate 0.001, weight decay 0.0001, image dimension 2048, batch size 1, and at most 100 epochs. The pinned segmentation library trains object locations, class scores, and mask pixels with its implemented loss functions. train.py records all explicit settings.

This is a new training recipe for study. The submitted predictions load the stored released weights. Training has a separate compute and storage cost, so RUN_NEW_TRAINING starts as False. Check available storage and GPU memory before starting a training run.

In [ ]:
RUN_NEW_TRAINING = False
if RUN_NEW_TRAINING:
    import train
    data = train.prepare(TRAIN, OUTPUT / "new-training")
    trained = train.train_from_scratch(data, architecture_index=0, device="0", epochs=100)

## How to extend the method
Use the fixed diagnostic examples to inspect boundaries, missed filaments, extra detections, and merged instances. For a new model, use an observation-grouped validation set to measure generalization. Keep preprocessing and mask-emission settings recorded alongside each model.

The original model cards identify the 707 sanctioned training observations. provenance.json records recovered training arguments and the limit of the independently available history. Our stored copies preserve their original attribution and exact file identity.